# Pilot Training — Deteksi Sarkasme pada Dataset SARC

Sebelum melatih model dengan data penuh, kita coba dulu dengan data yang lebih kecil. Tahap ini disebut **pilot**, dan sesuai proposal tujuannya ada dua:

1. Memastikan semua kode berjalan dari awal sampai akhir tanpa error.
2. Menentukan *batch size* yang muat di memori GPU.

Sekalian, pilot dipakai untuk mencoba tiga nilai *learning rate* supaya training utama nanti tidak perlu menebak-nebak.

**Alur kerja:**

```
data/tokenized (train_pilot 40.000 + val 20.000)
   → Uji batch size: berapa memori GPU yang terpakai?
   → Latih RoBERTa & DistilBERT dengan 3 learning rate
   → Nilai di data validasi + ukur waktu, memori, dan kecepatan prediksi
   → Simpan hasil
```

**Dijalankan di:** Google Colab, GPU Tesla T4
**Input:** `tokenized/roberta/` dan `tokenized/distilbert/` (file `train_pilot.parquet` dan `val.parquet`)
**Output:** `pilot_batch_size_probe.csv`, `pilot_history.csv`, `pilot_efficiency.csv`, `pilot_config.json`

> **Penting:** angka di notebook ini adalah hasil uji coba, bukan hasil penelitian. Hasil yang dilaporkan nanti berasal dari training utama dengan data penuh di GPU RTX 5070 Ti.

## 1. Hubungkan Google Drive

Data hasil tokenisasi disimpan di Google Drive, jadi Drive perlu dihubungkan dulu ke Colab. Baris kodenya dijadikan komentar karena saat notebook ini dijalankan Drive sudah terhubung. Kalau membuka sesi Colab baru, hapus tanda `#` lalu jalankan.

In [12]:
# from google.colab import drive
# drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Setup & Konfigurasi

| Konfigurasi | Nilai | Keterangan |
|---|---|---|
| `DEBUG` | False | Kalau `True`, notebook hanya memakai 300 data untuk mengecek kode dengan cepat |
| `LEARNING_RATES` | 1e-5, 2e-5, 3e-5 | Tiga nilai yang dicoba untuk kedua model |
| `NUM_EPOCHS` | 3 | Model membaca seluruh data latih sebanyak 3 kali |
| `BATCH_SIZE` | 32 | Jumlah data yang diproses sekaligus dalam satu langkah |
| `WARMUP_RATIO` | 0,1 | 10% langkah pertama dipakai untuk menaikkan learning rate pelan-pelan |
| `WEIGHT_DECAY` | 0,01 | Membantu mengurangi overfitting |
| `USE_FP16` | True | *Mixed precision*: hitungan memakai angka 16-bit supaya lebih cepat dan hemat memori (proposal) |

Kedua model memakai pengaturan yang sama persis. Dengan begitu, kalau hasilnya berbeda, penyebabnya memang modelnya, bukan pengaturannya.

In [13]:
import gc, json, math, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

DEBUG = False                              # True = uji cepat dengan data sangat kecil

SEED = 42
MODELS = {
    "roberta": "roberta-base",
    "distilbert": "distilbert-base-uncased",
}
LEARNING_RATES = [1e-5, 2e-5, 3e-5]
NUM_EPOCHS = 3
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
PROBE_BATCH_SIZES = [16, 32, 64, 128]     # batch size yang dicoba pada uji memori

DRIVE_DIR = Path("/content/drive/MyDrive/Dataset")
DATA_DIR = DRIVE_DIR / "tokenized"
OUT_DIR = DRIVE_DIR / "experiments/results/pilot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()      # FP16 hanya tersedia di GPU

if DEBUG:
    LEARNING_RATES, NUM_EPOCHS, PROBE_BATCH_SIZES = [2e-5], 1, [16, 32]


## 3. Cek GPU

In [14]:
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
print("Device:", DEVICE, "| FP16:", USE_FP16)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name, "| VRAM:", round(gpu.total_memory / 1024**3, 1), "GB")
else:
    print("PERINGATAN: GPU tidak terdeteksi. Pilih Runtime > Change runtime type > GPU.")


torch: 2.11.0+cu128 | transformers: 5.16.1
Device: cuda | FP16: True
GPU: Tesla T4 | VRAM: 14.6 GB


#### Insight
- GPU yang dipakai adalah **Tesla T4** dengan memori (VRAM) **14,6 GB**, dan FP16 aktif.
- GPU ini berbeda dari RTX 5070 Ti (16 GB) yang dipakai untuk eksperimen utama. Jadi angka waktu dan memori di notebook ini hanya gambaran kasar dan perlu diukur ulang di sana.

## 4. Load Data Tokenized

Data dibaca dari hasil notebook `03_tokenization`. Tidak ada tokenisasi ulang di sini.

Satu hal yang perlu dijelaskan adalah **dynamic padding**. Model butuh semua input dalam satu batch sama panjang, jadi input yang pendek ditambah token kosong (*padding*). Daripada menyamakan semua input ke 256 token, padding dilakukan per batch: cukup sepanjang input terpanjang di batch itu. Karena kebanyakan input hanya sekitar 32 token, cara ini jauh lebih hemat waktu.

In [15]:
class PairDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.input_ids = [x.tolist() for x in df["input_ids"]]
        self.attention_mask = [x.tolist() for x in df["attention_mask"]]
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i],
                "attention_mask": self.attention_mask[i],
                "labels": self.labels[i]}

def load_split(model_key, split):
    df = pd.read_parquet(DATA_DIR / model_key / f"{split}.parquet")
    if DEBUG:
        df = df.groupby("label", group_keys=False).sample(150, random_state=SEED)
    return df.reset_index(drop=True)

frames = {key: {"train": load_split(key, "train_pilot"), "val": load_split(key, "val")} for key in MODELS}
tokenizers = {key: AutoTokenizer.from_pretrained(name) for key, name in MODELS.items()}
collators = {key: DataCollatorWithPadding(tok) for key, tok in tokenizers.items()}   # dynamic padding

for key in MODELS:
    print(f"{key:10s} train_pilot: {len(frames[key]['train']):,} | val: {len(frames[key]['val']):,}")


roberta    train_pilot: 40,000 | val: 20,000
distilbert train_pilot: 40,000 | val: 20,000


#### Insight
- Data latih pilot berisi **40.000 pasangan** dan data validasi **20.000 pasangan**, sama untuk kedua model.
- Data validasi yang dipakai sama dengan yang dipakai di training utama, sedangkan data test belum disentuh sama sekali.

## 5. Fungsi Metrik & Pengukur Efisiensi

Bagian ini menyiapkan alat ukur. Belum ada yang dijalankan.

**Metrik performa** (kelas positif = sarkas):

| Metrik | Arti sederhana |
|---|---|
| Accuracy | Dari semua tebakan, berapa persen yang benar |
| Precision | Dari yang ditebak sarkas, berapa persen yang memang sarkas |
| Recall | Dari semua yang memang sarkas, berapa persen yang berhasil ditemukan |
| F1-score | Gabungan precision dan recall. Ini metrik utama di proposal |

**Metrik efisiensi:**

| Metrik | Arti sederhana |
|---|---|
| Jumlah parameter | Ukuran model |
| Waktu training | Lama melatih model |
| Memori puncak | Memori GPU terbanyak yang terpakai selama training |
| Latensi | Waktu yang dibutuhkan model untuk memprediksi satu data |

Saat mengukur latensi, beberapa batch pertama tidak dihitung, karena GPU butuh "pemanasan" dan hasil awalnya biasanya lebih lambat dari yang sebenarnya.

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(labels, preds),
            "precision": precision, "recall": recall, "f1": f1}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def peak_memory_gb():
    return torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else float("nan")

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

@torch.no_grad()
def measure_latency(model, dataset, collator, batch_size, n_batches=50, n_warmup=5):
    """Rata-rata waktu inferensi per sampel (milidetik)."""
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, collate_fn=collator)
    times, n_samples = [], 0
    for step, batch in enumerate(loader):
        if step >= n_batches + n_warmup:
            break
        batch = {k: v.to(model.device) for k, v in batch.items() if k != "labels"}
        sync()
        start = time.perf_counter()
        with torch.autocast(device_type="cuda", enabled=USE_FP16):
            model(**batch)
        sync()
        if step >= n_warmup:                      # batch awal tidak dihitung (pemanasan)
            times.append(time.perf_counter() - start)
            n_samples += len(batch["input_ids"])
    return sum(times) / n_samples * 1000


## 6. Uji Batch Size

Pertanyaannya: *batch size* berapa yang masih muat di memori GPU?

Caranya, setiap model dilatih beberapa langkah dengan batch berisi **input-input terpanjang** (256 token semua). Ini kasus terburuk. Kalau kasus terburuk saja muat, training yang sebenarnya pasti aman.

In [17]:
def probe_batch_size(model_key, batch_size, n_steps=10):
    """Latih beberapa langkah dengan batch terpanjang (kasus terburuk) lalu catat memori GPU."""
    free_memory()
    df = frames[model_key]["train"].nlargest(batch_size, "n_tokens")
    ds = PairDataset(df)
    batch = collators[model_key]([ds[i] for i in range(len(ds))])
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    result = {"Model": model_key, "Batch size": batch_size, "Panjang batch": batch["input_ids"].shape[1]}
    model = optimizer = None
    try:
        model = AutoModelForSequenceClassification.from_pretrained(MODELS[model_key], num_labels=2).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
        scaler = torch.amp.GradScaler(enabled=USE_FP16)

        model.train()
        n_warmup = 2                                   # langkah awal tidak dihitung (pemanasan GPU)
        for step in range(n_steps + n_warmup):
            if step == n_warmup:
                sync()
                start = time.perf_counter()
            with torch.autocast(device_type="cuda", enabled=USE_FP16):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        sync()

        result.update({"Status": "OK", "Memori puncak (GB)": round(peak_memory_gb(), 2),
                       "Detik per langkah": round((time.perf_counter() - start) / n_steps, 3)})
    except torch.cuda.OutOfMemoryError:
        result.update({"Status": "OOM (memori tidak cukup)"})
    del model, optimizer, batch
    free_memory()
    return result

probe = pd.DataFrame([probe_batch_size(key, bs) for key in MODELS for bs in PROBE_BATCH_SIZES])
display(probe)
probe.to_csv(OUT_DIR / "pilot_batch_size_probe.csv", index=False)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,Model,Batch size,Panjang batch,Status,Memori puncak (GB),Detik per langkah
0,roberta,16,256,OK,3.17,0.251
1,roberta,32,256,OK,4.71,0.469
2,roberta,64,256,OK,7.79,0.959
3,roberta,128,256,OK,13.97,1.539
4,distilbert,16,256,OK,1.66,0.111
5,distilbert,32,256,OK,2.45,0.196
6,distilbert,64,256,OK,4.02,0.362
7,distilbert,128,256,OK,7.18,0.713


> **Catatan:** tabel `LOAD REPORT` yang muncul berulang kali itu normal. `UNEXPECTED` adalah lapisan lama yang tidak dipakai, dan `MISSING` adalah lapisan klasifikasi baru yang memang belum terlatih. Lapisan baru inilah yang dilatih untuk membedakan sarkas dan non-sarkas.

#### Insight

| Batch size | RoBERTa | DistilBERT |
|---:|---:|---:|
| 16 | 3,17 GB | 1,66 GB |
| 32 | 4,71 GB | 2,45 GB |
| 64 | 7,79 GB | 4,02 GB |
| 128 | 13,97 GB | 7,18 GB |

- **Semua ukuran batch muat**, tidak ada yang kehabisan memori.
- **DistilBERT butuh kira-kira setengah memori RoBERTa** di setiap ukuran batch.
- **RoBERTa dengan batch 128 terlalu mepet**: 13,97 GB dari 14,6 GB yang tersedia. Sedikit saja ada beban tambahan, training bisa gagal.
- **Batch besar tidak membuat training lebih cepat.** Pada RoBERTa, kecepatannya tetap sekitar 64–68 data per detik untuk batch 16, 32, dan 64. Jadi tidak ada untungnya memaksakan batch besar.
- **Keputusan: batch size 32 untuk kedua model.** Memorinya masih sangat longgar dan ukuran ini umum dipakai untuk fine-tuning. Angka ini akan dicek ulang di RTX 5070 Ti.

## 7. Fungsi Pilot Training

Fungsi `run_pilot` melatih satu model dengan satu nilai learning rate, lalu mencatat hasilnya. Urutannya:

1. Muat model dari awal (bobot *pre-trained*).
2. Latih selama 3 epoch.
3. Di akhir setiap epoch, nilai model pada data validasi.
4. Catat waktu training, memori puncak, dan latensi.

Seed dibuat sama (`SEED = 42`) di setiap percobaan, supaya perbedaan hasil hanya berasal dari model dan learning rate.

In [18]:
def run_pilot(model_key, learning_rate):
    set_seed(SEED)
    free_memory()
    train_ds = PairDataset(frames[model_key]["train"])
    val_ds = PairDataset(frames[model_key]["val"])
    total_steps = math.ceil(len(train_ds) / BATCH_SIZE) * NUM_EPOCHS

    model = AutoModelForSequenceClassification.from_pretrained(MODELS[model_key], num_labels=2)
    args = TrainingArguments(
        output_dir=f"/content/pilot_{model_key}_{learning_rate}",
        learning_rate=learning_rate,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=int(WARMUP_RATIO * total_steps),
        fp16=USE_FP16,
        eval_strategy="epoch",
        save_strategy="no",                       # pilot tidak menyimpan checkpoint
        logging_strategy="epoch",
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=collators[model_key], compute_metrics=compute_metrics)

    sync()
    start = time.perf_counter()
    trainer.train()
    sync()
    train_time = time.perf_counter() - start
    peak_mem = peak_memory_gb()

    logs = trainer.state.log_history
    train_loss = {round(l["epoch"]): l["loss"] for l in logs if "loss" in l}
    rows = [{
        "Model": model_key, "Learning rate": learning_rate, "Epoch": round(l["epoch"]),
        "Train loss": train_loss.get(round(l["epoch"])), "Val loss": l["eval_loss"],
        "Accuracy": l["eval_accuracy"], "Precision": l["eval_precision"],
        "Recall": l["eval_recall"], "F1": l["eval_f1"],
    } for l in logs if "eval_f1" in l]

    efficiency = {
        "Model": model_key, "Learning rate": learning_rate,
        "Jumlah parameter (juta)": round(count_parameters(model) / 1e6, 1),
        "Waktu training (menit)": round(train_time / 60, 2),
        "Memori puncak training (GB)": round(peak_mem, 2),
        "Latensi batch=1 (ms/sampel)": round(measure_latency(model, val_ds, collators[model_key], 1), 2),
        "Latensi batch=32 (ms/sampel)": round(measure_latency(model, val_ds, collators[model_key], 32), 3),
    }
    del model, trainer
    free_memory()
    return rows, efficiency


## 8. Jalankan Pilot

Total ada **6 percobaan**: 2 model × 3 learning rate. Hasil disimpan ke Drive setiap satu percobaan selesai, jadi kalau Colab terputus di tengah jalan, hasil yang sudah ada tidak hilang.

In [19]:
history, efficiency = [], []
for model_key in MODELS:
    for lr in LEARNING_RATES:
        print(f"\n===== {model_key} | learning rate = {lr} =====")
        rows, eff = run_pilot(model_key, lr)
        history += rows
        efficiency.append(eff)
        pd.DataFrame(history).to_csv(OUT_DIR / "pilot_history.csv", index=False)        # simpan bertahap
        pd.DataFrame(efficiency).to_csv(OUT_DIR / "pilot_efficiency.csv", index=False)

history = pd.DataFrame(history)
efficiency = pd.DataFrame(efficiency)



===== roberta | learning rate = 1e-05 =====


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.616528,0.555676,0.715950,0.775763,0.607500,0.681398
2,0.526363,0.568666,0.723650,0.703448,0.773300,0.736722
3,0.475212,0.559478,0.730850,0.725682,0.742300,0.733897



===== roberta | learning rate = 2e-05 =====


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.620696,0.571223,0.700300,0.735425,0.625700,0.676140
2,0.513309,0.579499,0.710650,0.698633,0.740900,0.719146
3,0.417961,0.610975,0.712550,0.715677,0.705300,0.710451



===== distilbert | learning rate = 3e-05 =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.615481,0.566412,0.703250,0.742745,0.621900,0.676972
2,0.491508,0.580956,0.710050,0.689320,0.764800,0.725101
3,0.352654,0.656964,0.712550,0.710009,0.718600,0.714279


#### Insight
- Keenam percobaan selesai tanpa error. Total waktunya sekitar **83 menit**.
- Satu percobaan RoBERTa butuh ±18 menit, sedangkan DistilBERT ±9 menit.
- Di semua percobaan, *training loss* turun dari epoch ke epoch. Artinya model memang belajar dari data.

## 9. Hasil Performa

Tabel pertama berisi hasil setiap epoch dari keenam percobaan. Tabel kedua berisi hasil terbaik tiap model berdasarkan F1 di data validasi.

In [20]:
def show(df):
    display(df.style.format({"Learning rate": "{:.0e}"}, precision=4))

show(history)

best = history.loc[history.groupby("Model")["F1"].idxmax()].set_index("Model")
print("Konfigurasi terbaik per model (berdasarkan F1 validasi):")
show(best)


,Model,Learning rate,Epoch,Train loss,Val loss,Accuracy,Precision,Recall,F1
0,roberta,1e-05,1,0.6165,0.5557,0.7159,0.7758,0.6075,0.6814
1,roberta,1e-05,2,0.5264,0.5687,0.7237,0.7034,0.7733,0.7367
2,roberta,1e-05,3,0.4752,0.5595,0.7308,0.7257,0.7423,0.7339
3,roberta,2e-05,1,0.6129,0.5563,0.7216,0.7686,0.6339,0.6948
4,roberta,2e-05,2,0.5101,0.5778,0.7235,0.6992,0.7843,0.7393
5,roberta,2e-05,3,0.4277,0.5889,0.7326,0.7210,0.7588,0.7394
6,roberta,3e-05,1,0.6100,0.5516,0.7248,0.7557,0.6642,0.7070
7,roberta,3e-05,2,0.4973,0.5716,0.7261,0.6984,0.7959,0.7440
8,roberta,3e-05,3,0.3895,0.6101,0.7367,0.7280,0.7557,0.7416
9,distilbert,1e-05,1,0.6340,0.5864,0.6872,0.7064,0.6406,0.6719


Konfigurasi terbaik per model (berdasarkan F1 validasi):


,Learning rate,Epoch,Train loss,Val loss,Accuracy,Precision,Recall,F1
Model,,,,,,,,
distilbert,3e-05,2,0.4915,0.5810,0.7100,0.6893,0.7648,0.7251
roberta,3e-05,2,0.4973,0.5716,0.7261,0.6984,0.7959,0.7440


#### Insight

**Ringkasan F1 terbaik tiap percobaan:**

| Learning rate | RoBERTa | DistilBERT |
|---|---:|---:|
| 1e-5 | 0,7367 | 0,7013 |
| 2e-5 | 0,7394 | 0,7191 |
| 3e-5 | **0,7440** | **0,7251** |

- **Kedua model berhasil belajar.** Accuracy-nya 69–74%, jauh di atas 50% (hasil kalau model hanya menebak asal).
- **RoBERTa lebih akurat** di semua learning rate. Pada hasil terbaik, selisih F1-nya sekitar 2 poin (0,744 vs 0,725).
- **Learning rate 3e-5 memberi hasil terbaik untuk kedua model.** Tapi selisihnya dengan 2e-5 kecil sekali (sekitar 0,005), jadi jangan dianggap pasti lebih baik. Dengan satu kali percobaan, selisih sekecil itu bisa saja kebetulan.
- **Model mulai overfitting setelah epoch ke-2.** Tandanya: *training loss* terus turun, tetapi *validation loss* malah naik. Contohnya RoBERTa dengan 3e-5: training loss turun dari 0,61 ke 0,39, sedangkan validation loss naik dari 0,55 ke 0,61. Model makin hafal data latih, tapi tidak makin pintar di data baru.
- **Hasil terbaik hampir selalu ada di epoch 2, bukan epoch 3.** Karena itu, di training utama model yang dipakai sebaiknya model dari epoch terbaik, bukan dari epoch terakhir.
- **Precision dan recall naik-turun antar-epoch.** Di epoch 1 model cenderung "hati-hati" (precision tinggi, recall rendah). Di epoch 2 model lebih berani menebak sarkas (recall naik, precision turun). F1 dipakai sebagai metrik utama karena menyeimbangkan keduanya.

## 10. Hasil Efisiensi

Tabel ini berisi biaya komputasi dari keenam percobaan.

In [21]:
display(efficiency)


,Model,Learning rate,Jumlah parameter (juta),Waktu training (menit),Memori puncak training (GB),Latensi batch=1 (ms/sampel),Latensi batch=32 (ms/sampel)
0,roberta,0.00001,124.6,18.40,4.75,12.55,2.077
1,roberta,0.00002,124.6,18.39,4.75,17.93,2.071
2,roberta,0.00003,124.6,18.41,4.75,13.12,2.095
3,distilbert,0.00001,67.0,9.32,2.47,6.66,1.067
4,distilbert,0.00002,67.0,9.32,2.47,6.53,1.085
5,distilbert,0.00003,67.0,9.32,2.47,6.63,1.087


#### Insight

| Metrik | RoBERTa | DistilBERT | Perbandingan |
|---|---:|---:|---|
| Jumlah parameter | 124,6 juta | 67,0 juta | RoBERTa 1,86× lebih besar |
| Waktu training (3 epoch) | 18,4 menit | 9,3 menit | DistilBERT 1,97× lebih cepat |
| Memori puncak training | 4,75 GB | 2,47 GB | DistilBERT 1,92× lebih hemat |
| Latensi (batch 32) | ±2,08 ms per data | ±1,08 ms per data | DistilBERT 1,93× lebih cepat |
| Latensi (batch 1) | 12,6–17,9 ms | 6,5–6,7 ms | DistilBERT kira-kira 2× lebih cepat |

- **DistilBERT kira-kira dua kali lebih ringan di semua sisi**: ukuran, waktu training, memori, dan kecepatan prediksi.
- **Waktu training sangat stabil.** Tiga percobaan RoBERTa selesai dalam 18,39–18,41 menit, jadi cara pengukurannya bisa dipercaya.
- **Learning rate tidak memengaruhi efisiensi.** Waktu dan memori sama saja untuk ketiga nilai, karena jumlah langkah training-nya sama.
- **Latensi batch 1 pada RoBERTa belum stabil** (12,55 / 17,93 / 13,12 ms). Angka 17,93 kemungkinan gangguan sesaat di Colab. Di training utama, pengukuran ini perlu diulang lebih banyak dan diambil nilai tengahnya.

## 11. Simpan Konfigurasi

Semua pengaturan dan hasil terbaik disimpan ke `pilot_config.json`, supaya jelas pilot ini dijalankan dengan pengaturan apa dan di perangkat apa.

In [22]:
pilot_config = {
    "debug": DEBUG,
    "seed": SEED,
    "models": MODELS,
    "learning_rates": LEARNING_RATES,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "fp16": USE_FP16,
    "padding": "dynamic",
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(DEVICE),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "n_train": {k: len(v["train"]) for k, v in frames.items()},
    "n_val": {k: len(v["val"]) for k, v in frames.items()},
    "best": {m: {"learning_rate": r["Learning rate"], "epoch": int(r["Epoch"]), "f1": round(r["F1"], 4)}
             for m, r in best.iterrows()},
}
(OUT_DIR / "pilot_config.json").write_text(json.dumps(pilot_config, indent=2))
print(json.dumps(pilot_config, indent=2))


{
  "debug": false,
  "seed": 42,
  "models": {
    "roberta": "roberta-base",
    "distilbert": "distilbert-base-uncased"
  },
  "learning_rates": [
    1e-05,
    2e-05,
    3e-05
  ],
  "num_epochs": 3,
  "batch_size": 32,
  "warmup_ratio": 0.1,
  "weight_decay": 0.01,
  "fp16": true,
  "padding": "dynamic",
  "device": "Tesla T4",
  "torch_version": "2.11.0+cu128",
  "transformers_version": "5.16.1",
  "n_train": {
    "roberta": 40000,
    "distilbert": 40000
  },
  "n_val": {
    "roberta": 20000,
    "distilbert": 20000
  },
  "best": {
    "distilbert": {
      "learning_rate": 3e-05,
      "epoch": 2,
      "f1": 0.7251
    },
    "roberta": {
      "learning_rate": 3e-05,
      "epoch": 2,
      "f1": 0.744
    }
  }
}


## 12. Kesimpulan & Langkah Selanjutnya

### Kesimpulan
1. **Pipeline berjalan lancar.** Dari membaca data sampai menyimpan hasil, tidak ada error.
2. **Batch size 32 aman** untuk kedua model, dengan sisa memori yang masih banyak.
3. **Learning rate 3e-5 memberi F1 validasi terbaik** untuk RoBERTa (0,744) maupun DistilBERT (0,725).
4. **Tiga epoch sudah cukup.** Model mulai overfitting setelah epoch ke-2, jadi menambah epoch tidak akan membantu.
5. **Gambaran awal trade-off:** DistilBERT mempertahankan sekitar **97,5%** F1 RoBERTa (0,725 dari 0,744), dengan waktu training dan memori kira-kira **setengahnya**.

### Keterbatasan pilot
- Data latih hanya 40.000 pasangan, seperempat dari data latih penuh.
- Setiap percobaan hanya dijalankan satu kali dengan satu seed.
- Dijalankan di Tesla T4, bukan di RTX 5070 Ti.
- Dinilai pada data validasi, bukan data test.

Karena itu, angka-angka di atas dipakai untuk **menentukan pengaturan**, bukan untuk dilaporkan sebagai hasil penelitian.
